# setup

In [74]:
"""
matched_vs_unmatched_analysis.py

QUESTION THIS ANSWERS
----------------------
For each metric, we matched a subset of treatment (HN-submitted) repos to
control repos. This script checks whether the repos that got MATCHED look
systematically different from the ones that were AVAILABLE but not matched
-- i.e. a representativeness / balance diagnostic, done separately for the
treatment side and the control side.

This is *not* the parallel-trends check (that's your event-study code) --
it's a sanity check on sample selection: "does matching quietly select for
unusually big/small/active repos?"

SUMMARY STATISTIC USED PER REPO
--------------------------------
Each repo's monthly panel is collapsed to ONE number before plotting:
  - for a "new_X" (flow) metric  -> sum over the observed window
  - for a "cumulative_X" (stock) metric -> the max value in the window
    (its final accumulated total)
Per your caveat, cumulative_X is just a running sum of new_X, so these two
are conceptually the same "how much activity did this repo generate"
number -- we just pick the right aggregation for each column type so we
don't double count.

OUTPUTS
-------
plots/matched_vs_unmatched/<metric>_box_violin.png   (per metric)
plots/matched_vs_unmatched/<metric>_ecdf.png         (per metric)
plots/matched_vs_unmatched/love_plot_smd.png         (one summary figure)
plots/matched_vs_unmatched/balance_table.csv         (numbers behind the love plot)
"""

'\nmatched_vs_unmatched_analysis.py\n\nQUESTION THIS ANSWERS\n----------------------\nFor each metric, we matched a subset of treatment (HN-submitted) repos to\ncontrol repos. This script checks whether the repos that got MATCHED look\nsystematically different from the ones that were AVAILABLE but not matched\n-- i.e. a representativeness / balance diagnostic, done separately for the\ntreatment side and the control side.\n\nThis is *not* the parallel-trends check (that\'s your event-study code) --\nit\'s a sanity check on sample selection: "does matching quietly select for\nunusually big/small/active repos?"\n\nSUMMARY STATISTIC USED PER REPO\n--------------------------------\nEach repo\'s monthly panel is collapsed to ONE number before plotting:\n  - for a "new_X" (flow) metric  -> sum over the observed window\n  - for a "cumulative_X" (stock) metric -> the max value in the window\n    (its final accumulated total)\nPer your caveat, cumulative_X is just a running sum of new_X, so thes

In [75]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import textwrap

# ============================================================
# CONFIG -- double-check these five things before running
# ============================================================

DATASET_DIR = Path(".")
MATCH_DIR = Path(".")  # wherever matched_control_<metric>.csv live

# NOTE: going by your earlier notebook (match-parallel-trend-assumption.ipynb),
# "hn-stories-..." = TREATMENT (HN-submitted repos) and "nonhn-..." = CONTROL.
# Your message labeled them the other way round -- swap these two lines back
# if that was intentional and not a typo.
TREATMENT_METRICS_FILE = DATASET_DIR / "hn-stories-gh-ai-metrics-cumulative.csv"
CONTROL_METRICS_FILE = DATASET_DIR / "nonhn-gh-ai-metrics-cumulative.csv"

TREATMENT_REPO_COL = "repo_full_name"  # column identifying a repo in the treatment file
CONTROL_REPO_COL = "control_repo"      # column identifying a repo in the control file

MONTHLY_METRICS = ['new_forks', 'new_stars', 'new_commits', 'new_prs', 'active_contributors']
CUMULATIVE_METRICS = ['cumulative_stars', 'cumulative_forks', 'cumulative_commits',
                       'cumulative_prs', 'cumulative_ve_contributors']
ALL_METRICS = MONTHLY_METRICS + CUMULATIVE_METRICS

OUTPUT_DIR = Path("plots/matched_vs_unmatched")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

sns.set_theme(style="whitegrid", font_scale=1.05)

# Set global aesthetic settings
sns.set_theme(style="whitegrid")
plt.rcParams.update({'font.size': 10})


In [76]:
# ============================================================
# 1. LOAD DATA
# ============================================================

def load_metrics_panels():
    """Load the two long-format panels once, up front, so we don't
    re-read 150k-row CSVs inside a per-metric loop."""
    treatment_metrics = pd.read_csv(TREATMENT_METRICS_FILE)
    control_metrics = pd.read_csv(CONTROL_METRICS_FILE)
    return treatment_metrics, control_metrics


def load_matched_repo_sets(metric):
    """matched_control_<metric>.csv has one row per matched pair.
    We only need *which repos* were matched, as two sets."""
    path = MATCH_DIR / f"matched_control_{metric}.csv"
    df = pd.read_csv(path)
    return set(df['treatment_repo']), set(df['control_repo'])

In [77]:

# ============================================================
# 2. COLLAPSE EACH REPO'S PANEL TO ONE COMPARABLE NUMBER
# ============================================================

def summarize_repo_metric(metrics_df, repo_col, metric_col, is_cumulative):
    """One row per repo: total observed activity for `metric_col`
    across the whole panel window."""
    agg_func = 'max' if is_cumulative else 'sum'
    out = (metrics_df.groupby(repo_col)[metric_col]
                      .agg(agg_func)
                      .rename('total_value')
                      .reset_index()
                      .rename(columns={repo_col: 'repo'}))
    return out


def build_comparison_df(treatment_metrics, control_metrics, metric):
    """Tidy frame for one metric: repo | group | matched | total_value."""
    is_cumulative = metric in CUMULATIVE_METRICS
    matched_treat, matched_ctrl = load_matched_repo_sets(metric)

    treat = summarize_repo_metric(treatment_metrics, TREATMENT_REPO_COL, metric, is_cumulative)
    treat['group'] = 'treatment'
    treat['matched'] = treat['repo'].isin(matched_treat)

    ctrl = summarize_repo_metric(control_metrics, CONTROL_REPO_COL, metric, is_cumulative)
    ctrl['group'] = 'control'
    ctrl['matched'] = ctrl['repo'].isin(matched_ctrl)

    df = pd.concat([treat, ctrl], ignore_index=True)
    df['matched_label'] = df['matched'].map({True: 'matched', False: 'unmatched'})
    # log1p makes heavy-tailed count data readable without dropping zeros
    df['log1p_value'] = np.log1p(df['total_value'])
    return df


# box plot

In [78]:
# ============================================================
# 3. VISUAL 1 -- box + violin, side by side, per metric
# ============================================================

def plot_box_and_violin(df, metric):
    fig, axes = plt.subplots(1, 2, figsize=(11, 5))

    sns.boxplot(data=df, x='group', hue='matched_label',
                ax=axes[0], showfliers=True)
    axes[0].set_title(f"{metric} -- boxplot (log1p scale)")
    axes[0].set_ylabel("log1p(total activity)")

    sns.violinplot(data=df, x='group', hue='matched_label',
                    ax=axes[1], split=True, inner='quartile', cut=0)
    axes[1].set_title(f"{metric} -- violin (shows shape a boxplot hides)")
    axes[1].set_ylabel("log1p(total activity)")

    for ax in axes:
        ax.legend(title="")
    fig.tight_layout()
    fig.savefig(OUTPUT_DIR / f"{metric}_box_violin.png", dpi=300, bbox_inches='tight')
    plt.close(fig)


# ECDF

In [54]:


# ============================================================
# 4. VISUAL 2 -- ECDF, per metric, per group
# ============================================================

def ecdf(values):
    x = np.sort(values)
    y = np.arange(1, len(x) + 1) / len(x)
    return x, y

def plot_ecdf(df, metric):
    fig, axes = plt.subplots(1, 2, figsize=(11, 5), sharey=True)
    for ax, group in zip(axes, ['treatment', 'control']):
        sub = df[df['group'] == group]
        for label, color in [('matched', 'tab:blue'), ('unmatched', 'tab:orange')]:
            vals = sub.loc[sub['matched_label'] == label, 'log1p_value']
            if len(vals) == 0:
                continue
            x, y = ecdf(vals)
            ax.plot(x, y, label=f"{label} (n={len(vals)})", color=color, linewidth=2)
        ax.set_title(f"{metric} -- {group}")
        ax.set_xlabel("log1p(total activity)")
        ax.legend()
    axes[0].set_ylabel("cumulative proportion of repos")
    fig.tight_layout()
    fig.savefig(OUTPUT_DIR / f"{metric}_ecdf.png", dpi=150)
    plt.close(fig)


# love plot

In [79]:
# ============================================================
# 5. SUMMARY -- standardized mean difference + love plot
# ============================================================

def standardized_mean_diff(a, b):
    """Cohen's-d style SMD between two samples on the log1p scale.
    |SMD| < 0.1 is the common 'good balance' rule of thumb (Austin, 2011)."""
    a, b = np.asarray(a, dtype=float), np.asarray(b, dtype=float)
    pooled_std = np.sqrt((a.var(ddof=1) + b.var(ddof=1)) / 2)
    if pooled_std == 0 or np.isnan(pooled_std):
        return 0.0
    return (a.mean() - b.mean()) / pooled_std


def build_balance_table(treatment_metrics, control_metrics):
    rows = []
    for metric in ALL_METRICS:
        df = build_comparison_df(treatment_metrics, control_metrics, metric)
        for group in ['treatment', 'control']:
            sub = df[df['group'] == group]
            matched_vals = sub.loc[sub['matched'], 'log1p_value']
            unmatched_vals = sub.loc[~sub['matched'], 'log1p_value']
            rows.append({
                'metric': metric,
                'group': group,
                'n_matched': len(matched_vals),
                'n_unmatched': len(unmatched_vals),
                'mean_matched': matched_vals.mean() if len(matched_vals) else np.nan,
                'mean_unmatched': unmatched_vals.mean() if len(unmatched_vals) else np.nan,
                'smd': standardized_mean_diff(matched_vals, unmatched_vals),
            })
    return pd.DataFrame(rows)


def plot_love_plot(balance_df):
    fig, ax = plt.subplots(figsize=(8, 6))
    balance_df = balance_df.sort_values(['metric', 'group'])
    y_labels = [f"{m} ({g})" for m, g in zip(balance_df['metric'], balance_df['group'])]
    y_pos = np.arange(len(balance_df))

    colors = balance_df['group'].map({'treatment': 'tab:blue', 'control': 'tab:orange'})
    ax.scatter(balance_df['smd'], y_pos, c=colors, s=60, zorder=3)

    # show legend bottom left outside the graph
    ax.legend(handles=[
        plt.Line2D([0], [0], marker='o', color='w', markerfacecolor='tab:blue', markersize=10, label='treatment'),
        plt.Line2D([0], [0], marker='o', color='w', markerfacecolor='tab:orange', markersize=10, label='control')
    ], loc='lower left', bbox_to_anchor=(-0.5, -0.15))

    ax.axvline(0, color='black', linewidth=1)
    ax.axvline(0.1, color='red', linestyle='--', linewidth=1)
    ax.axvline(-0.1, color='red', linestyle='--', linewidth=1)
    ax.set_yticks(y_pos)
    ax.set_yticklabels(y_labels)
    ax.set_xlabel("Standardized Mean Difference (matched - unmatched)")
    ax.set_title("Balance check: matched vs. unmatched repos\n(dashed lines = |SMD| = 0.1 threshold)")
    ax.invert_yaxis()
    fig.tight_layout()
    fig.savefig(OUTPUT_DIR / "love_plot_smd.png", dpi=150)
    plt.close(fig)


In [80]:
# ============================================================
# MAIN
# ============================================================

if __name__ == "__main__":
    treatment_metrics, control_metrics = load_metrics_panels()

    for metric in ALL_METRICS:
        print(f"Processing {metric} ...")
        df = build_comparison_df(treatment_metrics, control_metrics, metric)
        plot_box_and_violin(df, metric)
        plot_ecdf(df, metric)

    balance_df = build_balance_table(treatment_metrics, control_metrics)
    balance_df.to_csv(OUTPUT_DIR / "balance_table.csv", index=False)
    plot_love_plot(balance_df)

    print(f"\nDone. Outputs in: {OUTPUT_DIR.resolve()}")
    print(balance_df[['metric', 'group', 'n_matched', 'n_unmatched', 'smd']]
          .to_string(index=False))

Processing new_forks ...
Processing new_stars ...
Processing new_commits ...
Processing new_prs ...
Processing active_contributors ...
Processing cumulative_stars ...
Processing cumulative_forks ...
Processing cumulative_commits ...
Processing cumulative_prs ...
Processing cumulative_ve_contributors ...

Done. Outputs in: /home/pmwatt/Source/SP2024-GPT/src/rq3/3c-causal-analysis/plots/matched_vs_unmatched
                    metric     group  n_matched  n_unmatched       smd
                 new_forks treatment         94           69 -0.382739
                 new_forks   control         44         2103  2.508284
                 new_stars treatment         64           99 -1.299074
                 new_stars   control         53         2094  3.313789
               new_commits treatment         41          122 -0.861066
               new_commits   control         22         2125  1.847558
                   new_prs treatment         41          122 -0.501439
                   new_